# Capa Silver - Dimensión de Distribuidores

## Librerias

In [1]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Variables

In [2]:
%run ../00-Modulos/n_modulo_variables.ipynb

Servicios:
 Warehouse  = s3a://iceberg/warehouse
 EndPoint   = http://localhost:9000
 Uri        = thrift://localhost:9083
Base de Datos Landing =  iceberg.landing
Base de Datos Bronze  =  iceberg.bronze
Base de Datos Silver  =  iceberg.silver
Base de Datos Gold    =  iceberg.gold


## Spark Session

In [3]:
spark = spark_session_hive("Lakehouse-Iceberg")

## Extracción

In [4]:
df_dealerships = spark.read.table(f'{vSchemaSilv}.dealerships')
df_dim_dealerships = spark.read.table(f'{vSchemaGold}.dim_dealerships')

## Transformaciones

In [5]:
# Obtener valor maximo de la llave subrogada
max_sk = df_dim_dealerships.select(f.coalesce(f.max(f.col("DEALERSHIP_SK")), f.lit(0))).first()[0]

# Obtener registros nuevos y calcula su llave subrogada
df_dealerships_ins = df_dealerships.alias("d") \
    .join(df_dim_dealerships.alias("dd"), (f.col("d.DEALERSHIP_ID") == f.col("dd.DEALERSHIP_ID")), "left_anti") \
    .withColumn("DEALERSHIP_SK", max_sk + f.row_number().over(Window.orderBy(f.col("DEALERSHIP_ID")))) \
    .select(
        f.col("DEALERSHIP_SK"),
        f.col("d.*")
    )

# Obtener registros a actualizar
df_dealerships_upd = df_dealerships.alias("d") \
    .join(df_dim_dealerships.alias("dd"), (f.col("d.DEALERSHIP_ID") == f.col("dd.DEALERSHIP_ID")), "inner") \
    .select(
        f.col("dd.DEALERSHIP_SK"),
        f.col("d.*")
    )

# Seleccion de columnas finales
df_source = df_dealerships_ins.unionByName(df_dealerships_upd)

## Carga

In [6]:
# Obtenemos variables para merge
target_table = f'{vSchemaGold}.dim_dealerships'
join_columns = ["DEALERSHIP_SK"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)